# Result comparison

Checks that a changed query returns the same result as the original one, using the baseline stored by `benchmark.py --save-baseline` (`toolkit/results.py`).

Uses **query 72 at 1GB**:
1. Run the original query with the default Spark settings (`config/default_spark.toml`) and save its result to `SCENARIOS_DIR/results_1GB`.
2. Run the changed query and compare its result against the stored one.
3. Sanity checks: tampered results must be detected.

In [ ]:
import sys

sys.path.insert(0, "../pipeline")
from config_loader import load_config
from constants import (
    DEFAULT_SPARK_CONFIG_PATH,
    get_benchmark_queries_dir,
    get_results_dir,
    get_scenario_paths,
)
from spark_session import apply_run_options, create_spark_session, register_tables_in_spark
from toolkit.queries import normalize_sql_for_spark
from toolkit.results import compare_results, save_results

SCALE_FACTOR = 1
QUERY_FILE = "query72.sql"

options = load_config(DEFAULT_SPARK_CONFIG_PATH)
_, parquet_dir, _ = get_scenario_paths(SCALE_FACTOR, options.spark.use_partition)
results_dir = get_results_dir(SCALE_FACTOR)

In [ ]:
spark = create_spark_session(
    "result_comparison",
    use_all_cores=options.spark.use_all_cores,
    tune_memory_allocation=options.spark.tune_memory_allocation,
)
spark.sparkContext.setLogLevel("ERROR")
apply_run_options(spark, options)
register_tables_in_spark(spark, parquet_dir, register_as=options.spark.register_tables_as)

In [ ]:
def read_statements(version: str) -> list[str]:
    """Statements of QUERY_FILE, prepared the same way as in benchmark.py."""
    path = f"{get_benchmark_queries_dir(SCALE_FACTOR, version)}/{QUERY_FILE}"
    with open(path, encoding="utf-8") as f:
        sql = normalize_sql_for_spark(f.read())
    return [statement.strip() for statement in sql.split(";") if statement.strip()]


def run(sql: str):
    """Rows and schema of a statement, as benchmark.py gets them."""
    df = spark.sql(sql)
    return df.collect(), df.schema

## 1. Save the baseline

Original query, default Spark settings. One Parquet per statement (`query72.parquet`, or `query14_part1.parquet` for files with several statements), with the real column names and types and the rows in their original order.

In [ ]:
original_statements = read_statements("original")
original = {}
for part, sql in enumerate(original_statements, start=1):
    rows, schema = run(sql)
    original[part] = (rows, schema)
    path = save_results(
        spark, rows, schema, results_dir, QUERY_FILE,
        part if len(original_statements) > 1 else None,
    )
    print(f"part {part}: {len(rows)} rows saved to {path}")

In [ ]:
spark.read.parquet(f"{results_dir}/query72.parquet").show(5, truncate=40)

## 2. Comparison (`toolkit/results.compare_results`)

Against the stored baseline, in order:

1. **Schema**: column names and types must be identical (`schema_mismatch`).
2. **Rows in order**: exact equality of the row lists, NULLs equal, duplicates counted (`match`).
3. If not, `exceptAll` both ways tells apart `order_differs` (same rows, different order) from `mismatch`, with sample rows from each side.

The new result goes through the same `results_dataframe` as the saved one, so duplicate column names are renamed the same way on both sides.

If nothing was saved for a statement, it returns `no_baseline`.

In [ ]:
def compare_with_baseline(rows, schema, query_file: str, statement_part: int | None = None) -> dict:
    return compare_results(spark, rows, schema, results_dir, query_file, statement_part)

## 3. Changed query vs baseline

The actual check: the changed query 72 must give the same result as the original.

In [ ]:
changed_statements = read_statements("changed")
assert len(changed_statements) == len(original_statements), "different number of statements"

for part, sql in enumerate(changed_statements, start=1):
    rows, schema = run(sql)
    statement_part = part if len(changed_statements) > 1 else None
    print(f"part {part}:", compare_with_baseline(rows, schema, QUERY_FILE, statement_part))

## 4. Sanity checks

The comparison must also catch differences. Each case tampers with the original result (no extra query runs) and checks the reported status.

In [ ]:
from pyspark.sql.types import StructField, StructType

rows, schema = original[1]
renamed = StructType(
    [StructField("description", schema.fields[0].dataType)] + schema.fields[1:]
)
changed_row = rows[0].asDict()
changed_row["total_cnt"] += 1

cases = {
    "same result": (rows, schema, "match"),
    "reversed order": (rows[::-1], schema, "order_differs"),
    "one row missing": (rows[:-1], schema, "mismatch"),
    "one value changed": ([tuple(changed_row.values())] + rows[1:], schema, "mismatch"),
    "column renamed": (rows, renamed, "schema_mismatch"),
}
for name, (case_rows, case_schema, expected) in cases.items():
    result = compare_with_baseline(case_rows, case_schema, QUERY_FILE)
    mark = "✅" if result["status"] == expected else "❌"
    print(f"{mark} {name}: {result['status']} (expected {expected})")

print(compare_with_baseline(rows, schema, "query999.sql"))  # expected no_baseline

In [ ]:
spark.stop()